# Excel-отчёты через ZEMI Arsenal и Guidance

Arsenal загружает и запускает Qwen 3.5 4B, `MarkItDown` переводит Excel в компактный Markdown, а внутренний механизм Guidance ограничивает генерацию Pydantic-схемой `Reports`. `zemi.exp`, прямое чтение TOML и внешняя GBNF здесь не используются.

## Подготовка и запуск Arsenal

In [ ]:
from zemi.playbook import Arsenal


arsenal = Arsenal("@comp/playbook.toml")
# arsenal.download()  # Раскомментируйте, чтобы заранее скачать все ресурсы.
arsenal.begin_playbook(
    stop_arsenal_before_begin=True,
    llama_router_mode=False,
)

assistant_config = (
    arsenal.llamas["primary"]
    .models["qwen"]
    .assistants["report_parser"]
)
assistant_config.clients.timeout = 300.0
model = assistant_config.clients.guidance
model.echo = False

## Компактная схема результата

Guidance принимает Pydantic-класс непосредственно в `guidance.json`. Схема компилируется внутренним механизмом Guidance и ограничивает допустимые токены во время генерации.

In [ ]:
from pydantic import BaseModel, ConfigDict, Field


class StrictModel(BaseModel):
    model_config = ConfigDict(extra="forbid")


class Transaction(StrictModel):
    date: str = Field(description="Дата в формате YYYY-MM-DD")
    article: str
    cost: float


class Report(StrictModel):
    source_file: str
    city: str
    export_date: str = Field(description="Дата в формате YYYY-MM-DD")
    manager: str
    transactions: list[Transaction]


class Reports(StrictModel):
    reports: list[Report]

## Excel → Markdown

In [ ]:
from IPython.display import Markdown, display
from markitdown import MarkItDown

from zemi import env


data_dir = env.path.comp / "data/case01"
excel_files = [data_dir / f"Отчет {number}.xlsx" for number in range(1, 4)]
converter = MarkItDown(enable_plugins=False)

excel_context = "\n\n".join(
    f"# Файл: {path.name}\n\n{converter.convert(path).text_content.strip()}"
    for path in excel_files
)

print("Подготовленные файлы:")
for path in excel_files:
    print(f"- {path.name}: {path.stat().st_size} байт")
print(f"\nРазмер Markdown-контекста: {len(excel_context)} символов")
display(Markdown(excel_context))

## Структурированное извлечение Guidance

In [ ]:
import json

from guidance import assistant, json as gen_json, system, user


task = """
Обработай все переданные Excel-отчёты. Для каждого файла извлеки имя файла,
город/филиал, дату выгрузки, руководителя и строки таблицы (date, article, cost).
Не включай строку «Итого» и не выдумывай отсутствующие данные.
Все даты записывай строго в формате YYYY-MM-DD.
""".strip()

lm = model
with system():
    lm += (
        "Ты аккуратно преобразуешь Excel-отчёты в строго "
        "структурированные данные и ничего не выдумываешь."
    )
with user():
    lm += f"{task}\n\n{excel_context}"
with assistant():
    lm += gen_json(
        name="reports_json",
        schema=Reports,
        temperature=0.0,
        max_tokens=2048,
    )

result = Reports.model_validate_json(lm["reports_json"])
print(json.dumps(result.model_dump(mode="json"), ensure_ascii=False, indent=2))

In [ ]:
result

## Остановка Arsenal

Выполни эту ячейку, когда модель больше не нужна.

In [ ]:
arsenal.end_playbook(stop_arsenal_after_end=True)